In [1]:
import pandas as pd

df = pd.read_csv('player_stats.csv')
df.head()

,player,position,rating,total_remates,xG,passes_certos,toques,toques_area_adv,dribles,duelos,match_id,home,away,score
0,Larrazabal G.,Médio ofensivo,8.3,1,0.31,8/17 (47%),46,3,0/2 (0%),5,IcJBYh6e,Casa Pia AC,Torreense,2-0
1,Kaly,Defesa-central,8.1,1,0.21,17/18 (94%),38,-,1/1 (100%),11,IcJBYh6e,Casa Pia AC,Torreense,2-0
2,Ofori L.,Médio,7.8,1,0.91,16/18 (89%),29,2,-,5,IcJBYh6e,Casa Pia AC,Torreense,2-0
3,Prieto K.,Ala,7.8,2,0.03,13/18 (72%),45,1,1/2 (50%),14,IcJBYh6e,Casa Pia AC,Torreense,2-0
4,Geraldes A.,Ala,7.6,-,-,23/28 (82%),56,-,2/3 (67%),7,IcJBYh6e,Casa Pia AC,Torreense,2-0


In [2]:
import re

def parse_ratio(val):
    if pd.isna(val) or val == '-':
        return float('nan')
    m = re.search(r'\((\d+)%\)', str(val))
    return float(m.group(1)) if m else float('nan')

df.replace('-', float('nan'), inplace=True)

df['passes_certos_pct'] = df['passes_certos'].apply(parse_ratio)
df['dribles_pct']       = df['dribles'].apply(parse_ratio)
df['duelos_pct']        = df['duelos'].apply(parse_ratio)

df['xG']            = pd.to_numeric(df['xG'], errors='coerce')
df['total_remates'] = pd.to_numeric(df['total_remates'], errors='coerce')
df['rating']        = pd.to_numeric(df['rating'], errors='coerce')
df['toques']        = pd.to_numeric(df['toques'], errors='coerce')
df['toques_area_adv'] = pd.to_numeric(df['toques_area_adv'], errors='coerce')

df[['passes_certos_pct', 'dribles_pct', 'duelos_pct', 'xG', 'rating']].head()


,passes_certos_pct,dribles_pct,duelos_pct,xG,rating
0,47.0,0.0,NaN,0.31,8.3
1,94.0,100.0,NaN,0.21,8.1
2,89.0,NaN,NaN,0.91,7.8
3,72.0,50.0,NaN,0.03,7.8
4,82.0,67.0,NaN,NaN,7.6


In [ ]:
def score_to_result(score):
    h, a = map(int, score.split('-'))
    if h > a:   return 'H'
    elif h < a: return 'A'
    else:       return 'D'

df['result']       = df['score'].apply(score_to_result)
df['position_enc'] = df['position'].astype('category').cat.codes

print(df['result'].value_counts())
df[['player', 'position', 'position_enc', 'result']].head()


In [ ]:
from sklearn.model_selection import GroupShuffleSplit

feature_cols = [
    'rating', 'xG', 'total_remates',
    'passes_certos_pct', 'toques', 'toques_area_adv',
    'dribles_pct', 'duelos_pct', 'position_enc'
]

X      = df[feature_cols].fillna(0)
y      = df['result']
groups = df['match_id']

# split por match_id para evitar data leakage (jogadores do mesmo jogo só num split)
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups))
X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

print(f"Train: {len(X_train)} jogadores | Test: {len(X_test)} jogadores")
print(f"Jogos no teste: {df['match_id'].iloc[test_idx].nunique()}")


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
import matplotlib.pyplot as plt

models = {
    'Logistic Regression': LogisticRegression(max_iter=2000, random_state=42),
    'Random Forest':       RandomForestClassifier(n_estimators=200, random_state=42),
}

for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    print(f"=== {name} ===")
    print(f"Accuracy: {accuracy_score(y_test, preds):.3f}")
    print(classification_report(y_test, preds))


In [ ]:
try:
    from xgboost import XGBClassifier
    from sklearn.preprocessing import LabelEncoder

    le = LabelEncoder()
    y_train_enc = le.fit_transform(y_train)
    y_test_enc  = le.transform(y_test)

    xgb = XGBClassifier(n_estimators=200, random_state=42, eval_metric='mlogloss', verbosity=0)
    xgb.fit(X_train, y_train_enc)
    preds_xgb = le.inverse_transform(xgb.predict(X_test))

    print("=== XGBoost ===")
    print(f"Accuracy: {accuracy_score(y_test, preds_xgb):.3f}")
    print(classification_report(y_test, preds_xgb))
except ImportError:
    print("xgboost não instalado. Corre: pip install xgboost")


In [ ]:
rf_model = models['Random Forest']
importances = pd.Series(rf_model.feature_importances_, index=feature_cols).sort_values(ascending=False)

plt.figure(figsize=(8, 5))
importances.plot(kind='barh')
plt.title('Feature Importance — Random Forest (stats individuais)')
plt.xlabel('Importância')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()


In [ ]:
# Perfil médio histórico de cada jogador
player_profiles = (
    df.groupby('player')[feature_cols]
    .mean()
    .reset_index()
)

print(f"{len(player_profiles)} jogadores únicos no dataset")
player_profiles.head()


In [ ]:
def predict_match(home_players, away_players, model=None):
    """
    home_players: lista com 11 nomes (equipa da casa)
    away_players: lista com 11 nomes (equipa de fora)
    Devolve probabilidades H/D/A em percentagem.
    """
    if model is None:
        model = models['Random Forest']

    rows, not_found = [], []
    for name in home_players + away_players:
        profile = player_profiles[player_profiles['player'] == name]
        if profile.empty:
            not_found.append(name)
            rows.append({f: 0.0 for f in feature_cols})
        else:
            rows.append(profile[feature_cols].fillna(0).iloc[0].to_dict())

    if not_found:
        print(f"Jogadores não encontrados (usando médias 0): {not_found}")

    probas   = model.predict_proba(pd.DataFrame(rows)[feature_cols])
    avg_prob = probas.mean(axis=0)
    result   = {cls: round(p * 100, 1) for cls, p in zip(model.classes_, avg_prob)}

    print(f"\nCasa:  {', '.join(home_players)}")
    print(f"Fora:  {', '.join(away_players)}")
    print(f"\n  Vitória Casa (H): {result.get('H', 0):.1f}%")
    print(f"  Empate       (D): {result.get('D', 0):.1f}%")
    print(f"  Vitória Fora (A): {result.get('A', 0):.1f}%")
    return result


# --- Exemplo com jogadores reais do dataset ---
home_11 = player_profiles['player'].sample(11, random_state=1).tolist()
away_11 = player_profiles['player'].sample(11, random_state=2).tolist()

predict_match(home_11, away_11)
